# Picking one answer with `scorio.aggregate`

We use [Scorio](https://github.com/mohsenhariri/scorio) for scoring, ranking and
aggregating stochastic model responses. This notebook covers `scorio.aggregate`, which
takes a pool of sampled candidates for a question and returns the one answer you keep.

The input is two aligned arrays of shape `M x N`, questions by candidates. `answers`
holds the extracted final answer of each candidate, compared by equality. `scores` holds
one number per candidate, higher is better, from a verifier or from the trace's own
confidence. Selection rules return only the selection, so you score them yourself with
`scorio.eval`.

|  |  |
| --- | --- |
| module | [`scorio/aggregate`](https://github.com/mohsenhariri/scorio/tree/main/scorio/aggregate) |
| method reference | [`scorio/aggregate/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/README.md) |
| paper | [Test-Time Scaling in Reasoning LLMs: Inference Regimes, Evaluation, and Reproducibility](https://arxiv.org/abs/2608.04001) |
| video | [walkthrough](https://github.com/user-attachments/assets/01d7ab2f-6f1e-4ea5-9b02-189339aa1fdf) |
| docs | [scorio.readthedocs.io/en/latest/api/aggregate](https://scorio.readthedocs.io/en/latest/api/aggregate.html) |
| install | `pip install scorio` |

The data comes from the [Scorio Trace](https://huggingface.co/datasets/harimo/scorio-trace) dataset, see [trace.ipynb](https://github.com/mohsenhariri/scorio/blob/main/notebooks/datasets/trace/trace.ipynb).
One model on one task gives 30 pools of 80 candidates each.

In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from scorio import agg, eval

repo_name = "harimo/scorio-trace"
model_name, task = "Phi-4-reasoning", "aime25"

rows = (load_dataset(repo_name, "meta", split=task)
        .flatten()  # exposes tokens.completion_avg_logprob as a column
        .select_columns(["model_key", "data_id", "seed", "extracted_answer",
                         "ground_truth_accepted", "cv7b_label", "cv7b_prob",
                         "tokens.completion_avg_logprob"])
        .to_pandas())
rows = rows[rows.model_key == model_name].sort_values(["data_id", "seed"])

M, N = 30, 80

# what the model answered, with unparsed answers marked invalid so scorio ignores them
answers = rows.extracted_answer.to_numpy().reshape(M, N).astype(object)
answers[answers == "NotFound"] = None

# score 1, a reward model: P(correct) from CompassVerifier-7B
verifier = np.where(rows.cv7b_label.to_numpy() == "A", rows.cv7b_prob, 1 - rows.cv7b_prob).reshape(M, N)

# score 2, the trace's own confidence: mean log-probability of its tokens
confidence = rows["tokens.completion_avg_logprob"].to_numpy().reshape(M, N)

accepted = [set(a) for a in rows.groupby("data_id").ground_truth_accepted.first()]

print(answers.shape, verifier.shape, confidence.shape)

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

(30, 80) (30, 80) (30, 80)


## One question, eight candidates

Question 10 with the first eight seeds. Three candidates never produced a boxed answer and
are `None`, so scorio ignores them. Of the five that remain, 259 appears three times and is
the right answer, and both rules find it. The first sample alone would have answered 267.

In [2]:
pool, scores = answers[10, :8], verifier[10, :8]

print("candidates    ", list(pool))
print("verifier      ", scores.round(3))
print()
print("first sample  ", pool[0])
print("majority_vote ", agg.majority_vote(pool))
print("best_of_n     ", agg.best_of_n(pool, scores))
print("accepted      ", accepted[10])

candidates     ['267', '259', None, '22', '259', '259', None, None]
verifier       [0.001 0.999 0.    0.    0.999 0.999 0.999 0.042]

first sample   267
majority_vote  259
best_of_n      259
accepted       {'259'}


## All 30 questions, eight candidates each

Selection rules do not estimate their own accuracy. Look up whether the selected answer is
right, then hand that to `scorio.eval.bayes_ci` for a Bayes@N score and interval. This is
how the three modules fit together.

At this budget the verifier is worth more than the vote. Bayes@N moves from 0.567 for the
first sample to 0.578 with majority voting and 0.622 for the reward-model rules.

The sigma is the same for every row because it comes from the same 30 questions with one
decision each. With 30 questions these differences are suggestive, not settled.

In [3]:
def bayes_at_n(selected):
    """Bayes@N score for one selected answer per question."""
    hit = np.array([[1 if s in accepted[i] else 0] for i, s in enumerate(selected)])
    mu, sigma, lo, hi = eval.bayes_ci(hit)
    return {"Bayes@N": round(mu, 3), "sigma": round(sigma, 3),
            "lo": round(lo, 3), "hi": round(hi, 3)}


n = 8
A, V, C = answers[:, :n], verifier[:, :n], confidence[:, :n]

results = {
    "first sample": bayes_at_n(A[:, 0]),
    "majority_vote": bayes_at_n(agg.majority_vote(A)),
    "weighted_majority_vote (verifier)": bayes_at_n(agg.weighted_majority_vote(A, V)),
    "best_of_n (verifier)": bayes_at_n(agg.best_of_n(A, V)),
    "best_of_majority (verifier)": bayes_at_n(agg.best_of_majority(A, V)),
    "filtered_vote (logprob)": bayes_at_n(agg.filtered_vote(A, C, keep=0.5, weighted=False)),
    "rank_weighted_vote (logprob)": bayes_at_n(agg.rank_weighted_vote(A, C)),
}

display(pd.DataFrame(results).T.sort_values("Bayes@N", ascending=False))

,Bayes@N,sigma,lo,hi
weighted_majority_vote (verifier),0.622,0.043,0.538,0.707
best_of_majority (verifier),0.622,0.043,0.538,0.707
best_of_n (verifier),0.622,0.043,0.538,0.707
filtered_vote (logprob),0.611,0.043,0.527,0.695
rank_weighted_vote (logprob),0.589,0.043,0.505,0.673
majority_vote,0.578,0.043,0.493,0.662
first sample,0.567,0.043,0.482,0.651


Note the `weighted=False` on `filtered_vote` and the choice of `rank_weighted_vote` for the
log-probability score. Mean log-probability is negative, and a rule that sums scores as vote
weights needs non-negative ones. Rules that only use the ordering of the scores are safe
with any scale.

## Spending more samples

Majority voting improves as the sample budget grows, then levels off. The reward model
reaches that level with fewer samples and continues to improve at larger budgets.

In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]

sweep = pd.DataFrame({
    n: {
        "majority_vote": bayes_at_n(agg.majority_vote(answers[:, :n]))["Bayes@N"],
        "weighted (verifier)": bayes_at_n(agg.weighted_majority_vote(answers[:, :n], verifier[:, :n]))["Bayes@N"],
        "best_of_n (verifier)": bayes_at_n(agg.best_of_n(answers[:, :n], verifier[:, :n]))["Bayes@N"],
    } for n in budgets
}).T
sweep.index.name = "samples"

display(sweep)

,majority_vote,weighted (verifier),best_of_n (verifier)
samples,,,
1,0.567,0.567,0.567
2,0.589,0.611,0.611
4,0.567,0.622,0.622
8,0.578,0.622,0.622
16,0.600,0.622,0.622
32,0.600,0.644,0.644
80,0.600,0.644,0.644


## Stopping early

Both stopping rules watch the answers as they arrive and say when another sample is not
going to change the vote. They reach the same Bayes@N score as voting on all 80 candidates while
drawing a quarter to a sixth as many.

In [5]:
def run_until_stop(should_stop):
    """Draw candidates one at a time per question until the rule says stop."""
    used, selected = [], []
    for i in range(M):
        stop = N
        for t in range(2, N + 1):
            if should_stop(answers[i, :t]):
                stop = t
                break
        used.append(stop)
        selected.append(agg.majority_vote(answers[i, :stop]))
    return round(float(np.mean(used)), 1), bayes_at_n(selected)["Bayes@N"]


print("all 80 samples       ", (80.0, bayes_at_n(agg.majority_vote(answers))["Bayes@N"]))
print("adaptive_consistency ", run_until_stop(lambda seen: agg.adaptive_consistency_stop(seen, threshold=0.95)))
print("esc (window of 3)    ", run_until_stop(lambda seen: len(seen) >= 3 and agg.esc_stop(seen[-3:])))

all 80 samples        (80.0, 0.6)
adaptive_consistency  (21.4, 0.6)
esc (window of 3)     (13.7, 0.6)


## Confidence from the token log-probabilities

The `meta` config stores `completion_avg_logprob` per trace, which is what this notebook
used as the confidence score. The per-model configs carry the full per-token lists, and
`scorio.aggregate` computes the signals from them. Same numbers, as a check.

Signals that need top-k log-probabilities, such as `self_certainty`, `deepconf_confidence`
and `token_entropy`, are not reproducible from this dataset. Only the realized token's
log-probability and rank were stored.

In [6]:
one_pool = load_dataset(repo_name, model_name, split=task).select(range(10 * 80, 10 * 80 + 3))

for trace in one_pool:
    logprobs = trace["tokens"]["completion_logprob_list"]
    print(f"seed {trace['seed']}  "
          f"mean_logprob {agg.mean_logprob(logprobs):.6f} (stored {trace['tokens']['completion_avg_logprob']:.6f})  "
          f"perplexity {agg.perplexity(logprobs):.4f} (stored {trace['tokens']['completion_ppl']:.4f})")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

seed 1234  mean_logprob -0.155022 (stored -0.155022)  perplexity 1.1677 (stored 1.1677)
seed 1235  mean_logprob -0.206562 (stored -0.206562)  perplexity 1.2294 (stored 1.2294)
seed 1236  mean_logprob -0.279797 (stored -0.279797)  perplexity 1.3229 (stored 1.3229)


## Other APIs

Confidence signals in
[`confidence.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/confidence.py),
voting rules in [`vote.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/vote.py),
reward-based selection in [`best_of.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/best_of.py),
early stopping in [`online.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/online.py),
KDE calibration in [`calibration.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/calibration.py),
confidence-guided selection in [`cges.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/cges.py),
and process reward reduction in [`prm.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/prm.py).

Most published methods are a (signal, rule) pair. DeepConf offline voting is
`weighted_majority_vote` fed `deepconf_confidence`, self-certainty Best-of-N is `best_of_n`
fed `self_certainty`. The full table with a reference for each method is in
[`scorio/aggregate/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/README.md).